##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

In [3]:
# fine-tuning a EfficientNetB0 model on CIFAR-10
#   1) Load CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]
#   Keep labels as integers (SparseCategoricalCrossentropy)
y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")

#   Convert images to float32
x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")

#   2) Data augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")

#   3) Build EfficientNetB0 backbone (pretrained)
efficient_base = EfficientNetB0(
    include_top=False, 
    weights="imagenet",
    input_shape=(224, 224, 3)
)

#   4) Full model (preprocess inside model)
efficient_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Resizing(224, 224, interpolation="bilinear"),
    layers.Lambda(preprocess_input),          
    efficient_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10)                     
], name="cifar10_efficientnetb0")

efficient_model.summary()

# 5) Compile + Train 

efficient_base.trainable = True
for layer in efficient_base.layers[:-30]: # unfreeze last 30 layers
    layer.trainable = False

print("Trainable layers in backbone:", sum(l.trainable for l in efficient_base.layers), "/", len(efficient_base.layers))

efficient_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_ft = efficient_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)

test_loss_ft, test_acc_ft = efficient_model.evaluate(x_test, y_test, verbose=0)
print("EfficientNetB0 (fine-tuned) test accuracy:", test_acc_ft)
print("EfficientNetB0 (fine-tuned) test loss    :", test_loss_ft)


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "cifar10_efficientnetb0"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,062,381 (15.50 MB)

 Trainable params: 4,020,358 (15.34 MB)

 Non-trainable params: 42,023 (164.16 KB)

Trainable layers in backbone: 30 / 238
Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 138s 171ms/step - accuracy: 0.3433 - loss: 1.9791 - val_accuracy: 0.7952 - val_loss: 0.8627
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 118s 167ms/step - accuracy: 0.7176 - loss: 1.0262 - val_accuracy: 0.8562 - val_loss: 0.5095
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 118s 167ms/step - accuracy: 0.7757 - loss: 0.7274 - val_accuracy: 0.8814 - val_loss: 0.3966
EfficientNetB0 (fine-tuned) test accuracy: 0.8776999711990356
EfficientNetB0 (fine-tuned) test loss    : 0.4098442792892456


In [ ]:
#   - Network (EfficientNetB0) depth = 238 layers
#   - Total params: 4,062,381 (15.50 MB)
#   - 30 layers unfrozen (trainable) = 208 frozen

In [ ]:
# Compare with ResNet50V2 and custom CNN from previous lab?

# ResNet50V2 (fine-tuned) test accuracy: 0.91 - highest accuracy
# EfficientNetB0 (fine-tuned) test accuracy: 0.87
# Custom CNN (from lab1) test accuracy: 0.70
# it shows that ResNet50V2 is better than EfficientNetB0 on CIFAR-10, 
# but EfficientNetB0 is still much better than the custom CNN.

In [ ]:
# whih model is faster to train? 
# custom CNN is much faster to train than both ResNet50V2 and EfficientNetB0 
# since it has much fewer layers and parameters. 

In [ ]:
# Architectural differences:
# ResNet50V2 uses residual connections, while EfficientNetB0 uses compound scaling that uniformly 
# scales depth, width, and resolution. custom CNN has a much simpler architecture with fewer layers 
# and parameters compared to both ResNet50V2 and EfficientNetB0.